In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
# from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor,GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import mlflow
from sklearn.metrics import classification_report, mean_squared_error, r2_score, accuracy_score,silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans,DBSCAN,AgglomerativeClustering
import numpy as np


In [5]:
!pip install mlflow


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
df = pd.read_csv("reggession_clean_data.csv")

In [7]:
df2 = pd.read_csv("classification_clean_data.csv")

In [8]:
df['price_2'] = df2["price_2"]

In [9]:
df.head()

,page_2_clothing_model,colour,page_1_main_category,location,model_photography,session_id,country,date,order_level,price,price_2
0,A13,1,1,5,1,1,29,2008-04-01,Low,28,2
1,A16,1,1,6,1,1,29,2008-04-01,Low,33,2
2,B4,10,2,2,1,1,29,2008-04-01,Low,52,1
3,B17,6,2,6,2,1,29,2008-04-01,Low,38,2
4,B8,4,2,3,2,1,29,2008-04-01,Low,52,1


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165474 entries, 0 to 165473
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   page_2_clothing_model  165474 non-null  str  
 1   colour                 165474 non-null  int64
 2   page_1_main_category   165474 non-null  int64
 3   location               165474 non-null  int64
 4   model_photography      165474 non-null  int64
 5   session_id             165474 non-null  int64
 6   country                165474 non-null  int64
 7   date                   165474 non-null  str  
 8   order_level            165474 non-null  str  
 9   price                  165474 non-null  int64
 10  price_2                165474 non-null  int64
dtypes: int64(8), str(3)
memory usage: 16.4 MB


In [11]:
def preprocess_data(df):
    df.columns = df.columns.str.strip()
    df.dropna(subset=["price"], inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df.drop('date', axis=1, inplace=True)
    return df

In [12]:
data = preprocess_data(df)

In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 165474 entries, 0 to 165473
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   page_2_clothing_model  165474 non-null  str  
 1   colour                 165474 non-null  int64
 2   page_1_main_category   165474 non-null  int64
 3   location               165474 non-null  int64
 4   model_photography      165474 non-null  int64
 5   session_id             165474 non-null  int64
 6   country                165474 non-null  int64
 7   order_level            165474 non-null  str  
 8   price                  165474 non-null  int64
 9   price_2                165474 non-null  int64
 10  year                   165474 non-null  int32
 11  month                  165474 non-null  int32
 12  day                    165474 non-null  int32
dtypes: int32(3), int64(8), str(2)
memory usage: 15.4 MB


In [14]:
x_reg = data.drop(["price","price_2"],axis=1)
y_reg = data['price']
x_cls = data.drop("price_2",axis=1)
y_cls = data['price_2'] 
y_cls = data["price_2"].replace({2:0})

In [15]:
y_cls.value_counts()

price_2
1    84695
0    80779
Name: count, dtype: int64

In [16]:
categorical_cols = ["page_2_clothing_model","order_level"]
numerical_cols = ["colour","page_1_main_category","location","model_photography","session_id","country","year","month","day"]

In [17]:
numerical_transformer = Pipeline(
    steps=[
        ("encoder",StandardScaler())
    ]
)

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [18]:
preprocessor = ColumnTransformer(transformers=[("num",numerical_transformer,numerical_cols), ("cat",categorical_transformer,categorical_cols)])

In [19]:
y_cls.value_counts()

price_2
1    84695
0    80779
Name: count, dtype: int64

In [20]:
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(x_cls, y_cls, test_size=0.2, random_state=42)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(x_reg, y_reg, test_size=0.2, random_state=42)


In [21]:
df["price_2"]

0         2
1         2
2         1
3         2
4         1
         ..
165469    1
165470    1
165471    2
165472    1
165473    1
Name: price_2, Length: 165474, dtype: int64

In [22]:
X_cls_train.head()

,page_2_clothing_model,colour,page_1_main_category,location,model_photography,session_id,country,order_level,price,year,month,day
106979,C20,13,3,1,2,15648,29,Low,48,2008,6,22
69437,B26,13,2,3,1,10018,29,Low,57,2008,5,19
132136,C13,9,3,5,1,19388,29,Low,48,2008,7,15
50042,B11,2,2,4,1,7181,29,Low,43,2008,5,2
92387,B31,9,2,5,1,13493,29,Low,57,2008,6,9


In [23]:
# Classifier models
classifiers = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'DecisionTree': DecisionTreeClassifier(),
    'RandomForest': RandomForestClassifier(),
    'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'NeuralNet': MLPClassifier(hidden_layer_sizes=(100,), max_iter=300)
}

In [24]:
# Regressor models
regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'GradientBoosting': GradientBoostingRegressor()
}

In [25]:
## MLOps Tracking
mlflow.set_experiment("Customer_Conversion_Analysis")

# Train classification models
for name, model in classifiers.items():
    with mlflow.start_run(run_name=f"Classifier_{name}"):
        clf_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        clf_pipeline.fit(X_cls_train, y_cls_train)
        y_pred = clf_pipeline.predict(X_cls_test)
        acc = accuracy_score(y_cls_test, y_pred)
        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(clf_pipeline, f"classifier_{name}_model")

2026/04/14 19:52:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:52:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 19:52:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:52:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [26]:
y_reg_train.head()

106979    48
69437     57
132136    48
50042     43
92387     57
Name: price, dtype: int64

In [27]:
# Train regression models
for name, model in regressors.items():
    with mlflow.start_run(run_name=f"Regressor_{name}"):
        reg_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])
        reg_pipeline.fit(X_reg_train, y_reg_train)
        y_pred = reg_pipeline.predict(X_reg_test)
        rmse = mean_squared_error(y_reg_test, y_pred)
        r2 = r2_score(y_reg_test, y_pred)
        mlflow.log_params(model.get_params())
        mlflow.log_metrics({"rmse": rmse, "r2": r2})
        mlflow.sklearn.log_model(reg_pipeline, f"regressor_{name}_model")


2026/04/14 19:53:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:53:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 19:53:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:53:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [28]:
# Preprocess and convert to dense array
X_clustering = preprocessor.fit_transform(x_reg)

In [29]:
X_clustering_dense = X_clustering.toarray() if hasattr(X_clustering, "toarray") else X_clustering

In [30]:
clustering_models = {
    'KMeans': KMeans(n_clusters=3, random_state=42),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
}

for name, model in clustering_models.items():
    with mlflow.start_run(run_name=f"{name}_clustering"):
        labels = model.fit_predict(X_clustering_dense)
        print(f"\n{name} Clustering Labels:\n", labels)

        # Evaluation Metrics
        if len(set(labels)) > 1 and -1 not in set(labels):  # Ensure valid for silhouette and DBI
            sil_score = silhouette_score(X_clustering_dense, labels)
            dbi_score = davies_bouldin_score(X_clustering_dense, labels)
        else:
            sil_score, dbi_score = None, None

        # WCSS only valid for KMeans
        if name == 'KMeans':
            wcss = model.inertia_
        else:
            wcss = None

        # Log metrics
        if sil_score is not None:
            mlflow.log_metric("silhouette_score", sil_score)
        if dbi_score is not None:
            mlflow.log_metric("davies_bouldin_index", dbi_score)
        if wcss is not None:
            mlflow.log_metric("wcss", wcss)

        # Log model (if supported)
        if hasattr(model, 'fit'):
            mlflow.sklearn.log_model(model, f"{name}_model")

        mlflow.set_tag("model_type", "clustering")
        mlflow.log_param("model_name", name)

print("All clustering models trained, evaluated, and tracked with MLflow.")


KMeans Clustering Labels:
 [2 2 2 ... 1 1 1]


2026/04/14 19:58:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:58:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 19:59:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:59:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p


DBSCAN Clustering Labels:
 [   0    1    2 ... 2207   -1   -1]


2026/04/14 19:59:20 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


All clustering models trained, evaluated, and tracked with MLflow.


In [31]:
import os
import pickle

os.makedirs("models", exist_ok=True)

def save_model(model, filename):
    with open(f"models/{filename}", "wb") as f:
        pickle.dump(model, f)

In [32]:
for name, model in classifiers.items():
    with mlflow.start_run(run_name=f"Classifier_{name}"):

        clf_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])

        clf_pipeline.fit(X_cls_train, y_cls_train)

        y_pred = clf_pipeline.predict(X_cls_test)
        acc = accuracy_score(y_cls_test, y_pred)

        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(clf_pipeline, f"classifier_{name}_model")

        # ✅ ADD THIS LINE
        save_model(clf_pipeline, f"{name}_pipeline.pkl")

2026/04/14 19:59:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:59:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 19:59:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 19:59:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [33]:
for name, model in regressors.items():
    with mlflow.start_run(run_name=f"Regressor_{name}"):

        reg_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])

        reg_pipeline.fit(X_reg_train, y_reg_train)

        y_pred = reg_pipeline.predict(X_reg_test)

        rmse = mean_squared_error(y_reg_test, y_pred)
        r2 = r2_score(y_reg_test, y_pred)

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({"rmse": rmse, "r2": r2})
        mlflow.sklearn.log_model(reg_pipeline, f"regressor_{name}_model")

        # ✅ ADD THIS LINE
        save_model(reg_pipeline, f"{name}_pipeline.pkl")

2026/04/14 20:00:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 20:00:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 20:00:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 20:00:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [34]:
for name, model in clustering_models.items():
    with mlflow.start_run(run_name=f"{name}_clustering"):

        model.fit(X_clustering_dense)

        # ✅ SAVE MODEL
        save_model(model, f"{name}_clustering.pkl")

In [35]:
import sklearn
print(sklearn.__version__)

1.8.0


In [36]:
"""import os

print(os.getcwd())
print(os.path.exists("models/gradient_boosting_regression.pkl"))"""

'import os\n\nprint(os.getcwd())\nprint(os.path.exists("models/gradient_boosting_regression.pkl"))'

class DBSCANWrapper:
    def __init__(self, model):
        self.model = model
    
    def predict(self, X):
        return self.model.fit_predict(X)

In [37]:
"""import pickle

pickle.dump(model, open("gradient_boosting_regression_new.pkl", "wb"))"""

'import pickle\n\npickle.dump(model, open("gradient_boosting_regression_new.pkl", "wb"))'

In [38]:
# X_clustering = preprocessor.fit_transform(x_reg)

In [39]:
"""clustering_models = {
    'KMeans': KMeans(n_clusters=3, random_state=42),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
}"""

"clustering_models = {\n    'KMeans': KMeans(n_clusters=3, random_state=42),\n    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),\n}"

In [40]:
"""for name, model in clustering_models.items():
    labels = model.fit_predict(X_clustering_dense)
    
    print(f"\n{name} Clustering Labels:\n", labels)

    # Evaluation Metrics
    if len(set(labels)) > 1 and -1 not in set(labels):
        sil_score = silhouette_score(X_clustering_dense, labels)
        dbi_score = davies_bouldin_score(X_clustering_dense, labels)
    else:
        sil_score, dbi_score = None, None

    # WCSS (only for KMeans)
    if name == 'KMeans':
        wcss = model.inertia_
    else:
        wcss = None

    print(f"{name} Results:")
    print("Silhouette Score:", sil_score)
    print("Davies-Bouldin Index:", dbi_score)
    print("WCSS:", wcss)"""

'for name, model in clustering_models.items():\n    labels = model.fit_predict(X_clustering_dense)\n\n    print(f"\n{name} Clustering Labels:\n", labels)\n\n    # Evaluation Metrics\n    if len(set(labels)) > 1 and -1 not in set(labels):\n        sil_score = silhouette_score(X_clustering_dense, labels)\n        dbi_score = davies_bouldin_score(X_clustering_dense, labels)\n    else:\n        sil_score, dbi_score = None, None\n\n    # WCSS (only for KMeans)\n    if name == \'KMeans\':\n        wcss = model.inertia_\n    else:\n        wcss = None\n\n    print(f"{name} Results:")\n    print("Silhouette Score:", sil_score)\n    print("Davies-Bouldin Index:", dbi_score)\n    print("WCSS:", wcss)'

In [41]:
"""import os

os.makedirs("models", exist_ok=True)"""

'import os\n\nos.makedirs("models", exist_ok=True)'

In [42]:
"""import pickle

# --- Save function ---
def save_model(model, filename):
    with open(f"models/{filename}", "wb") as file:
        pickle.dump(model, file)

# =========================
# PREPARE DATA (IMPORTANT)
# =========================
# Use your dataset
# df = pd.read_csv("your_dataset.csv")

X = df.drop(['price', 'price_2'], axis=1, errors='ignore')
y_reg = df['price']
y_cls = df['price_2']

y_cls = y_cls.replace({1: 0, 2: 1})

# Only numeric
X = X.select_dtypes(include=['int64', 'float64'])

# =========================
# TRAIN + SAVE MODELS
# =========================

# Regression
lr = LinearRegression().fit(X, y_reg)
save_model(lr, "linearregression.pkl")

gbr = GradientBoostingRegressor().fit(X, y_reg)
save_model(gbr, "gradient_boosting_regression.pkl")

lasso = Lasso().fit(X, y_reg)
save_model(lasso, "lasso_model_regression.pkl")

ridge = Ridge().fit(X, y_reg)
save_model(ridge, "ridge_model_regression.pkl")

# Classification
log_reg = LogisticRegression(max_iter=1000).fit(X, y_cls)
save_model(log_reg, "logistic_regression_classification.pkl")

rf = RandomForestClassifier().fit(X, y_cls)
save_model(rf, "random_forest_classification.pkl")

mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300).fit(X, y_cls)
save_model(mlp, "neural_net_classification.pkl")

xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss").fit(X, y_cls)
save_model(xgb_model, "xgboost_classification.pkl")

# Clustering
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
save_model(kmeans, "k_means_clustering.pkl")

dbscan = DBSCAN(eps=0.5, min_samples=5).fit(X)
save_model(dbscan, "dbscan_clustering.pkl")

print("✅ All models trained and saved correctly!")"""

'import pickle\n\n# --- Save function ---\ndef save_model(model, filename):\n    with open(f"models/{filename}", "wb") as file:\n        pickle.dump(model, file)\n\n# =========================\n# PREPARE DATA (IMPORTANT)\n# =========================\n# Use your dataset\n# df = pd.read_csv("your_dataset.csv")\n\nX = df.drop([\'price\', \'price_2\'], axis=1, errors=\'ignore\')\ny_reg = df[\'price\']\ny_cls = df[\'price_2\']\n\ny_cls = y_cls.replace({1: 0, 2: 1})\n\n# Only numeric\nX = X.select_dtypes(include=[\'int64\', \'float64\'])\n\n# =========================\n# TRAIN + SAVE MODELS\n# =========================\n\n# Regression\nlr = LinearRegression().fit(X, y_reg)\nsave_model(lr, "linearregression.pkl")\n\ngbr = GradientBoostingRegressor().fit(X, y_reg)\nsave_model(gbr, "gradient_boosting_regression.pkl")\n\nlasso = Lasso().fit(X, y_reg)\nsave_model(lasso, "lasso_model_regression.pkl")\n\nridge = Ridge().fit(X, y_reg)\nsave_model(ridge, "ridge_model_regression.pkl")\n\n# Classificat

In [43]:
"""pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(eval_metric="logloss"))
])

pipeline.fit(X_cls_train, y_cls_train)

import pickle
pickle.dump(pipeline, open("xgboost_pipeline.pkl", "wb"))"""

'pipeline = Pipeline([\n    (\'preprocessor\', preprocessor),\n    (\'model\', xgb.XGBClassifier(eval_metric="logloss"))\n])\n\npipeline.fit(X_cls_train, y_cls_train)\n\nimport pickle\npickle.dump(pipeline, open("xgboost_pipeline.pkl", "wb"))'

In [44]:
"""pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_reg_train, y_reg_train)

pickle.dump(pipeline, open("linear_pipeline.pkl", "wb"))"""

'pipeline = Pipeline([\n    (\'preprocessor\', preprocessor),\n    (\'model\', LinearRegression())\n])\n\npipeline.fit(X_reg_train, y_reg_train)\n\npickle.dump(pipeline, open("linear_pipeline.pkl", "wb"))'